# Étude G2 — Seuil de déclenchement

**Question G2 :** pour chaque niveau de la Météo des forêts, quelle part des jours a eu au moins un départ de feu dans le 77 ?

**Indicateur (fiche G2) :** taux de jours d'alerte sans feu = jours de niveau ≥ N sans feu / jours de niveau ≥ N.
**Garde-fou :** taux de feux partis les jours de niveau < N.

Règles déjà décidées :
- un départ de feu = une fiche BDIFF (décision 006) ;
- on utilise `niveau_j1`, la prévision faite la veille pour le lendemain (fiche G2) ;
- comparaison limitée à l'Île-de-France (décision 007).

Sources : `data/bronze/` (voir `data/bronze/SOURCES.md`).

## 1. Chargement

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BRONZE = Path('..') / 'data' / 'bronze'
DEP_IDF = ['75', '77', '78', '91', '92', '93', '94', '95']
DEP_BSPP = ['75', '92', '93', '94']   # nombre de feux inconnu dans la BDIFF (décision 007)

### Météo des forêts

Le format change chaque année : séparateur `,` ou `;`, colonne `nom_dep` ou `dep_nom`. On lit chaque fichier tel quel, puis on harmonise les noms de colonnes.

In [ ]:
fichiers_mdf = {
    2024: ',',
    2025: ',',
    2026: ';',
}

morceaux = []
for annee, sep in fichiers_mdf.items():
    df = pd.read_csv(BRONZE / 'meteo_des_forets' / f'mdf_{annee}.csv', sep=sep, dtype=str)
    df = df.rename(columns={'dep_nom': 'nom_dep'})
    df['annee_fichier'] = annee
    print(annee, df.shape, list(df.columns))
    morceaux.append(df)

mdf = pd.concat(morceaux, ignore_index=True)
mdf.head()

On met tout au même format :
- `date_publication` : la date est en UTC, on la passe en heure française ;
- `code_departement` : sur 2 caractères (`1` devient `01`, `2A` ne change pas) ;
- les niveaux en nombres entiers.

In [ ]:
mdf['date_publication'] = pd.to_datetime(mdf['date'], utc=True).dt.tz_convert('Europe/Paris')
mdf['code_departement'] = mdf['num_dep'].str.strip().str.zfill(2)
mdf['niveau_j1'] = mdf['niveau_j1'].astype(int)
mdf['niveau_j2'] = mdf['niveau_j2'].astype(int)

mdf[['date', 'date_publication', 'num_dep', 'code_departement', 'niveau_j1']].head()

### BDIFF

Séparateur `;`, et **2 lignes d'information avant l'en-tête** : on les saute.

In [ ]:
bdiff = pd.read_csv(BRONZE / 'bdiff' / 'incendies_idf_2024_2025.csv', sep=';', skiprows=2, dtype=str)
bdiff['date_alerte'] = pd.to_datetime(bdiff['Date de première alerte'])
bdiff['jour'] = bdiff['date_alerte'].dt.date
print(bdiff.shape)
bdiff[['Année', 'Numéro', 'Département', 'Nom de la commune', 'Date de première alerte', 'Surface parcourue (m2)', 'Nature']].head()

## 2. Contrôles avant de croiser

In [ ]:
# Un seul niveau par jour et par département ?
mdf['jour_publication'] = mdf['date_publication'].dt.date
doublons = mdf.duplicated(['jour_publication', 'code_departement']).sum()
print('Doublons (jour, département) dans la Météo des forêts :', doublons)

# Combien de jours par saison, et de quand à quand ?
mdf.groupby('annee_fichier')['jour_publication'].agg(['min', 'max', 'nunique'])

In [ ]:
# Doublons dans la BDIFF (Année + Numéro identifie un feu)
print('Doublons BDIFF :', bdiff.duplicated(['Année', 'Numéro']).sum())

# Feux par département et par année
pd.crosstab(bdiff['Département'], bdiff['Année'], margins=True)

In [ ]:
# Valeurs manquantes dans les colonnes utiles
bdiff[['Date de première alerte', 'Département', 'Code INSEE', 'Nature']].isna().sum()

## 3. Le jour concerné par le niveau

La Météo des forêts est publiée la veille : `niveau_j1` vaut pour **le lendemain** de la publication. C'est ce jour-là qu'on rapproche des feux.

⚠️ Déduit de la documentation (publication à 17 h pour J+1 et J+2), pas écrit noir sur blanc : à confirmer.

In [ ]:
mdf['date_concernee'] = (mdf['date_publication'].dt.normalize() + pd.Timedelta(days=1)).dt.date
mdf.groupby('annee_fichier')['date_concernee'].agg(['min', 'max'])

## 4. La table « un jour × un département » (Île-de-France)

C'est la table dont G2 a besoin. Pour chaque jour et chaque département : le niveau, le nombre de feux, et s'il y a eu au moins un feu.

On s'arrête à 2024-2025 : la BDIFF 2026 n'est pas encore publiée.

In [ ]:
feux_par_jour = (bdiff.groupby(['jour', 'Département']).size()
                 .rename('nb_feux').reset_index()
                 .rename(columns={'jour': 'date_concernee', 'Département': 'code_departement'}))

jours = mdf[mdf['code_departement'].isin(DEP_IDF) & mdf['annee_fichier'].isin([2024, 2025])]
jours = jours[['date_concernee', 'code_departement', 'nom_dep', 'niveau_j1', 'date_publication']]

table = jours.merge(feux_par_jour, on=['date_concernee', 'code_departement'], how='left')

# Pas de feu ce jour-là : 0. Sauf pour la BSPP, où on ne sait pas : on laisse vide.
table['nb_feux'] = table['nb_feux'].fillna(0)
table.loc[table['code_departement'].isin(DEP_BSPP), 'nb_feux'] = pd.NA
table['nb_feux'] = table['nb_feux'].astype('Int64')
table['jour_avec_feu'] = (table['nb_feux'] > 0).astype('boolean')

print(table.shape)
table.sort_values(['date_concernee', 'code_departement']).head(10)

In [ ]:
# Vérification : chaque feu de la saison se retrouve-t-il dans la table ?
t77 = table[table['code_departement'] == '77']
print('Feux du 77 rattachés à un jour de la saison :', int(t77['nb_feux'].sum()))
print('Feux du 77 dans la BDIFF (toute l\'année)  :', (bdiff['Département'] == '77').sum())

## 5. Pour chaque niveau, quelle part des jours a eu un feu ? (Seine-et-Marne)

In [ ]:
par_niveau = (t77.groupby('niveau_j1')
              .agg(jours=('date_concernee', 'size'),
                   jours_avec_feu=('jour_avec_feu', 'sum'),
                   feux=('nb_feux', 'sum')))
par_niveau['part_jours_avec_feu_%'] = (100 * par_niveau['jours_avec_feu'] / par_niveau['jours']).round(1)
par_niveau

In [ ]:
ax = par_niveau[['jours_avec_feu']].assign(
        jours_sans_feu=par_niveau['jours'] - par_niveau['jours_avec_feu']
     ).plot(kind='bar', stacked=True, figsize=(6, 4))
ax.set_xlabel('Niveau de la Météo des forêts (niveau_j1)')
ax.set_ylabel('Nombre de jours')
ax.set_title('Seine-et-Marne, saisons 2024 et 2025')
plt.tight_layout()

## 6. L'indicateur G2 et son garde-fou, pour chaque valeur de N

In [ ]:
lignes = []
feux_saison = int(t77['nb_feux'].sum())
for N in [2, 3, 4]:
    alerte = t77['niveau_j1'] >= N
    jours_alerte = int(alerte.sum())
    jours_alerte_sans_feu = int((alerte & ~t77['jour_avec_feu']).sum())
    feux_sous_seuil = int(t77.loc[~alerte, 'nb_feux'].sum())
    lignes.append({
        'N': N,
        'jours d\'alerte': jours_alerte,
        'jours d\'alerte sans feu': jours_alerte_sans_feu,
        'taux d\'alerte sans feu %': round(100 * jours_alerte_sans_feu / jours_alerte, 1) if jours_alerte else None,
        'feux les jours < N': feux_sous_seuil,
        'garde-fou : taux de feux sous le seuil %': round(100 * feux_sous_seuil / feux_saison, 1) if feux_saison else None,
    })
pd.DataFrame(lignes).set_index('N')

## 7. Les feux que l'indice ne voit pas

La Météo des forêts n'est publiée que de juin à septembre. Combien de feux du 77 partent en dehors ?

In [ ]:
f77 = bdiff[bdiff['Département'] == '77'].copy()
dans_saison = f77['jour'].isin(set(t77['date_concernee']))
print('Feux du 77 :', len(f77), '| pendant la saison de l\'indice :', int(dans_saison.sum()), '| hors saison :', int((~dans_saison).sum()))
f77.loc[~dans_saison, 'date_alerte'].dt.month.value_counts().sort_index().rename('feux hors saison par mois')

## 8. Ce qu'on retient

À compléter après lecture des résultats :
- …
- …

Limites à garder en tête : 2 saisons seulement, très peu de jours de niveau élevé dans le 77, rattachement niveau / jour déduit, feux de forêt seulement (décision 006).